In [18]:
import numpy as np
import numpy.testing as npt
from scipy import stats
from scipy.stats import t,ttest_ind
from scipy.stats import f
from scipy.stats import f_oneway
import statsmodels.api as sm
from statsmodels.regression._prediction import get_prediction
from statsmodels.stats.outliers_influence import OLSInfluence,MLEInfluence
from statsmodels.graphics.gofplots import qqplot_2samples
import pandas as pd
from patsy import dmatrices
from numpy.testing import assert_almost_equal, assert_allclose
import matplotlib.pyplot as plt
import seaborn as sns

import sys
from copy import deepcopy


# =====================================
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, r'C:\Users\TODO\Desktop\Abhi\AI\AI\Math\Hands-On\statemodelsStudy')
from olsRegressionAnalysis import dispAnalysisOfVariance, tableDispFormatt,getInvOfProductMat,\
                                  getRegressionEqn,\
                                  dispReghressionAnalysis,norm_scalling,getCorrelation,\
                                  get_variance_inflation_factors,goodnessOfFitTestOfParams

This example taken from book **Introductions to linear regressions analysis**

***- Note: Lack of fit test:***

TODO: Required more efforts on this topic 

In [19]:
path  = r"C:\Users\TODO\Desktop\Abhi\AI\AI\allDataSet\STAT501_Lesson05\STAT501_Lesson05\LackOfFitDataSet.txt"
df = pd.read_csv(path)

#print(df)

In [20]:
y, X = dmatrices(
                 formula_like = 'y ~ x', 
                 data=df,
                 return_type='dataframe'
                 )
res = sm.OLS(y, X).fit()
dfX = pd.DataFrame({'X':df['x'],'Y':df['y'],'Y_CAP':res.predict()})
print(dfX)

      X      Y      Y_CAP
0   1.0  10.84  15.344271
1   1.0   9.30  15.344271
2   2.0  16.35  17.474660
3   3.3  22.88  20.244167
4   3.3  24.35  20.244167
5   4.0  24.56  21.735439
6   4.0  25.86  21.735439
7   4.0  29.16  21.735439
8   4.7  24.59  23.226712
9   5.0  22.25  23.865829
10  5.6  25.90  25.144062
11  5.6  27.20  25.144062
12  5.6  25.61  25.144062
13  6.0  25.45  25.996218
14  6.0  26.56  25.996218
15  6.5  21.03  27.061413
16  6.9  21.46  27.913569


In [21]:
dispAnalysisOfVariance(
    res
)

=============================== Analysis of Variance =======================================
  Source Of Var  DegOfFreedom(DF)  Sum Of Square(SS)  Mean Square(MS)          F      FSig         P
0    Regression               1.0          237.47877       237.478770  14.241103  4.543077  0.001839
1      Residual              15.0          250.13383        16.675589        ---       ---       ---
2         Total              16.0          487.61260       254.154358        ---       ---       ---
r-square:  0.48702344815829124 rSqr-adj 0.452825011368844 rSqr-Predict
=============================== Regression Equation ========================================
=============================== Regression eqn =============================================
13.214  + 2.13 x 


,Source Of Var,DegOfFreedom(DF),Sum Of Square(SS),Mean Square(MS),F,FSig,P
0,Regression,1.0,237.47877,237.478770,14.241103,4.543077,0.001839
1,Residual,15.0,250.13383,16.675589,---,---,---
2,Total,16.0,487.61260,254.154358,---,---,---


In [22]:
print(df['y'])


0     10.84
1      9.30
2     16.35
3     22.88
4     24.35
5     24.56
6     25.86
7     29.16
8     24.59
9     22.25
10    25.90
11    27.20
12    25.61
13    25.45
14    26.56
15    21.03
16    21.46
Name: y, dtype: float64


In [23]:
newDf = dfX['X'].value_counts()
newDf = newDf[newDf>1]
dfX['PE'] = dfX['X'].map(newDf)
dfX = dfX.dropna()
tableDispFormatt('DoF')
ser = dfX['X'].value_counts()-1
DoF = ser.sum()
print('Pure Error DoF: ',DoF,'DoF of LOF',res.df_resid-DoF)

=============================== DoF ========================================================
Pure Error DoF:  7 DoF of LOF 8.0


In [24]:
print('Pure Error DoF: ',DoF,'DoF of LOF',res.df_resid-DoF)

Pure Error DoF:  7 DoF of LOF 8.0


### Example 2: Lack of fit test

In [25]:
path  = r"C:\Users\TODO\Desktop\Abhi\AI\AI\allDataSet\STAT501_Lesson05\STAT501_Lesson05\newaccounts.txt"
df = pd.read_csv(path)


In [26]:
#print(df)

y, X = dmatrices(
                 formula_like = 'New ~ Size', 
                 data=df,
                 return_type='dataframe'
                 )
res = sm.OLS(y, X).fit()

In [27]:
#print(res.summary())
tableDispFormatt('Regression eqn')
dfX = pd.DataFrame({'X':df['Size'],'Y':df['New'],'Y_CAP':res.predict()})

=============================== Regression eqn =============================================


In [28]:
print(dfX)
print(dfX['X'].value_counts())

      X    Y       Y_CAP
0    75   28   87.225131
1    75   42   87.225131
2   100  112   99.392670
3   100  136   99.392670
4   125  160  111.560209
5   125  150  111.560209
6   150  152  123.727749
7   175  156  135.895288
8   175  124  135.895288
9   200  124  148.062827
10  200  104  148.062827
X
75     2
100    2
125    2
175    2
200    2
150    1
Name: count, dtype: int64


In [29]:
dispAnalysisOfVariance(
    res
)

=============================== Analysis of Variance =======================================
  Source Of Var  DegOfFreedom(DF)  Sum Of Square(SS)  Mean Square(MS)         F      FSig         P
0    Regression               1.0        5141.338410      5141.338410  3.138882  5.117355  0.110213
1      Residual               9.0       14741.570681      1637.952298       ---       ---       ---
2         Total              10.0       19882.909091      6779.290708       ---       ---       ---
r-square:  0.25858079352339614 rSqr-adj 0.17620088169266235 rSqr-Predict
=============================== Regression Equation ========================================
=============================== Regression eqn =============================================
50.723  + 0.487 Size 


,Source Of Var,DegOfFreedom(DF),Sum Of Square(SS),Mean Square(MS),F,FSig,P
0,Regression,1.0,5141.338410,5141.338410,3.138882,5.117355,0.110213
1,Residual,9.0,14741.570681,1637.952298,---,---,---
2,Total,10.0,19882.909091,6779.290708,---,---,---


### TODO Perform Lack of fit test